# Homework

In this homework, we'll practice streaming with Kafka (Redpanda) and PyFlink.

We use Redpanda, a drop-in replacement for Kafka. It implements the same
protocol, so any Kafka client library works with it unchanged.

For this homework we will be using Green Taxi Trip data from October 2025:

- [green_tripdata_2025-10.parquet](https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-10.parquet)


## Setup

We'll use the same infrastructure from the workshop.

Follow the setup instructions: build the Docker image, start the services:

```bash
cd 07-streaming/workshop/
docker compose build
docker compose up -d
```

This gives us:

- Redpanda (Kafka-compatible broker) on `localhost:9092`
- Flink Job Manager at http://localhost:8081
- Flink Task Manager
- PostgreSQL on `localhost:5432` (user: `postgres`, password: `postgres`)

If you previously ran the workshop and have old containers/volumes,
do a clean start:

```bash
docker compose down -v
docker compose build
docker compose up -d
```

Note: the container names (like `workshop-redpanda-1`) assume the
directory is called `workshop`. If you renamed it, adjust accordingly.


## Question 1. Redpanda version

Run `rpk version` inside the Redpanda container:

```bash
docker exec -it workshop-redpanda-1 rpk version
```

What version of Redpanda are you running?

In [2]:
!docker exec 07-streaming-redpanda-1 rpk version

rpk version: v25.3.9
Git ref:     836b4a36ef6d5121edbb1e68f0f673c2a8a244e2
Build date:  2026 Feb 26 07 48 21 Thu
OS/Arch:     linux/amd64
Go version:  go1.24.3

Redpanda Cluster
  node-1  v25.3.9 - 836b4a36ef6d5121edbb1e68f0f673c2a8a244e2


## Question 2. Sending data to Redpanda

Create a topic called `green-trips`:

```bash
docker exec -it workshop-redpanda-1 rpk topic create green-trips
```

Now write a producer to send the green taxi data to this topic.

Read the parquet file and keep only these columns:

- `lpep_pickup_datetime`
- `lpep_dropoff_datetime`
- `PULocationID`
- `DOLocationID`
- `passenger_count`
- `trip_distance`
- `tip_amount`
- `total_amount`

Convert each row to a dictionary and send it to the `green-trips` topic.
You'll need to handle the datetime columns - convert them to strings
before serializing to JSON.

Measure the time it takes to send the entire dataset and flush:

```python
from time import time

t0 = time()

# send all rows ...

producer.flush()

t1 = time()
print(f'took {(t1 - t0):.2f} seconds')
```

How long did it take to send the data?

- 10 seconds
- 60 seconds
- 120 seconds
- 300 seconds

In [3]:
!docker exec 07-streaming-redpanda-1 rpk topic create green-trips

TOPIC        STATUS
green-trips  TOPIC_ALREADY_EXISTS: The topic has already been created


In [4]:
import dataclasses
import json
import time
import pandas as pd
from kafka import KafkaProducer
from models import ride_from_row

# Download NYC yellow taxi trip data (first 1000 rows)
url = "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-10.parquet"
columns = ['lpep_pickup_datetime', 
           'lpep_dropoff_datetime', 
           'PULocationID', 
           'DOLocationID', 
           'passenger_count', 
           'trip_distance', 
           'tip_amount', 
           'total_amount'
           ]
df = pd.read_parquet(url, columns=columns)
df.head()

,lpep_pickup_datetime,lpep_dropoff_datetime,PULocationID,DOLocationID,passenger_count,trip_distance,tip_amount,total_amount
0,2025-10-01 00:21:47,2025-10-01 00:24:37,247,69,1.0,0.70,1.70,10.00
1,2025-10-01 00:14:03,2025-10-01 00:24:14,66,25,1.0,1.61,2.78,16.68
2,2025-10-01 00:16:44,2025-10-01 00:16:47,244,244,1.0,0.00,2.20,13.20
3,2025-10-01 00:07:36,2025-10-01 00:32:14,95,170,1.0,10.37,11.31,67.85
4,2025-09-30 21:10:29,2025-09-30 21:22:30,82,138,1.0,4.07,6.82,34.12


In [5]:
def ride_serializer(ride):
    ride_dict = dataclasses.asdict(ride)
    json_str = json.dumps(ride_dict)
    return json_str.encode('utf-8')

server = 'localhost:9092'

producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=ride_serializer
)

In [6]:
t0 = time.time()

topic_name = 'green-trips'

for _, row in df.iterrows():
    ride = ride_from_row(row)
    producer.send(topic_name, value=ride)

producer.flush()

t1 = time.time()
print(f'took {(t1 - t0):.2f} seconds')

took 5.95 seconds


## Question 3. Consumer - trip distance

Write a Kafka consumer that reads all messages from the `green-trips` topic
(set `auto_offset_reset='earliest'`).

Count how many trips have a `trip_distance` greater than 5.0 kilometers.

How many trips have `trip_distance` > 5?

- 6506
- 7506
- 8506
- 9506

In [ ]:
from datetime import datetime
from kafka import KafkaConsumer
from models import ride_deserializer

server = 'localhost:9092'

consumer = KafkaConsumer(
    topic_name,
    bootstrap_servers=[server],
    auto_offset_reset='earliest',
    group_id='rides-console',
    value_deserializer=ride_deserializer
)

print(f"Listening to {topic_name}...")

count = 0
trips = 0
for message in consumer:
    ride = message.value

    if ride.trip_distance > 5:
        trips += 1

    count += 1
    if count == len(df):
        break

print(count, "messages received")
print(trips, "trips have `trip_distance` > 5")

consumer.close()

Listening to green-trips...
49416 messages received
8506 trips have `trip_distance` > 5


## Part 2: PyFlink (Questions 4-6)

For the PyFlink questions, you'll adapt the workshop code to work with
the green taxi data. The key differences from the workshop:

- Topic name: `green-trips` (instead of `rides`)
- Datetime columns use `lpep_` prefix (instead of `tpep_`)
- You'll need to handle timestamps as strings (not epoch milliseconds)

You can convert string timestamps to Flink timestamps in your source DDL:

```sql
lpep_pickup_datetime VARCHAR,
event_timestamp AS TO_TIMESTAMP(lpep_pickup_datetime, 'yyyy-MM-dd HH:mm:ss'),
WATERMARK FOR event_timestamp AS event_timestamp - INTERVAL '5' SECOND
```

Before running the Flink jobs, create the necessary PostgreSQL tables
for your results.

In [27]:
"""CREATE TABLE processed_events_aggregated (
    window_start TIMESTAMP,
    PULocationID INT,
    num_trips BIGINT,
    PRIMARY KEY (window_start, PULocationID)
);"""

'CREATE TABLE processed_events_aggregated (\n    window_start TIMESTAMP,\n    PULocationID INT,\n    num_trips BIGINT,\n    PRIMARY KEY (window_start, PULocationID)\n);'

Important notes for the Flink jobs:

- Place your job files in `workshop/src/job/` - this directory is
  mounted into the Flink containers at `/opt/src/job/`
- Submit jobs with:
  `docker exec -it workshop-jobmanager-1 flink run -py /opt/src/job/your_job.py`
- The `green-trips` topic has 1 partition, so set parallelism to 1
  in your Flink jobs (`env.set_parallelism(1)`). With higher parallelism,
  idle consumer subtasks prevent the watermark from advancing.
- Flink streaming jobs run continuously. Let the job run for a minute
  or two until results appear in PostgreSQL, then query the results.
  You can cancel the job from the Flink UI at http://localhost:8081
- If you sent data to the topic multiple times, delete and recreate
  the topic to avoid duplicates:
  `docker exec -it workshop-redpanda-1 rpk topic delete green-trips`


In [24]:
!docker exec 07-streaming-redpanda-1 rpk topic delete green-trips
!docker exec 07-streaming-redpanda-1 rpk topic create green-trips

TOPIC        STATUS
green-trips  OK
TOPIC        STATUS
green-trips  OK


In [25]:
for _, row in df.iterrows():
    ride = ride_from_row(row)
    producer.send(topic_name, value=ride)

producer.flush()

In [12]:
#!docker exec 07-streaming-jobmanager-1 flink run -py /opt/src/job/aggregation_job.py

## Question 4. Tumbling window - pickup location

Create a Flink job that reads from `green-trips` and uses a 5-minute
tumbling window to count trips per `PULocationID`.

Write the results to a PostgreSQL table with columns:
`window_start`, `PULocationID`, `num_trips`.

After the job processes all data, query the results:

```sql
SELECT PULocationID, num_trips
FROM <your_table>
ORDER BY num_trips DESC
LIMIT 3;
```

Which `PULocationID` had the most trips in a single 5-minute window?

- 42
- 74
- 75
- 166

In [9]:
import psycopg2

conn = psycopg2.connect(
    host="localhost",
    port=5432,
    dbname="postgres",
    user="postgres",
    password="postgres"
)

cur = conn.cursor()
cur.execute("""
    SELECT PULocationID, num_trips
    FROM processed_events_aggregated
    ORDER BY num_trips DESC
    LIMIT 3
""")

rows = cur.fetchall()
for row in rows:
    print(f"PULocationID: {row[0]}, num_trips: {row[1]}")

cur.close()
conn.close()

PULocationID: 74, num_trips: 15
PULocationID: 74, num_trips: 14
PULocationID: 74, num_trips: 13


## Question 5. Session window - longest streak

Create another Flink job that uses a session window with a 5-minute gap
on `PULocationID`, using `lpep_pickup_datetime` as the event time
with a 5-second watermark tolerance.

A session window groups events that arrive within 5 minutes of each other.
When there's a gap of more than 5 minutes, the window closes.

Write the results to a PostgreSQL table and find the `PULocationID`
with the longest session (most trips in a single session).

How many trips were in the longest session?

- 12
- 31
- 51
- 81

In [10]:
"""CREATE TABLE processed_events_session (
    window_start TIMESTAMP,
    window_end TIMESTAMP,
    PULocationID INT,
    num_trips BIGINT,
    PRIMARY KEY (window_start, window_end, PULocationID)
);"""

'CREATE TABLE processed_events_session (\n    window_start TIMESTAMP,\n    window_end TIMESTAMP,\n    PULocationID INT,\n    num_trips BIGINT,\n    PRIMARY KEY (window_start, window_end, PULocationID)\n);'

In [ ]:
#!docker exec 07-streaming-jobmanager-1 flink run -py /opt/src/job/session_job.py

In [26]:
conn = psycopg2.connect(
    host="localhost",
    port=5432,
    dbname="postgres",
    user="postgres",
    password="postgres"
)

cur = conn.cursor()
cur.execute("""
    SELECT PULocationID, num_trips, window_start, window_end
    FROM processed_events_session
    ORDER BY num_trips DESC
    LIMIT 3
""")

rows = cur.fetchall()
for row in rows:
    print(f"PULocationID: {row[0]}, num_trips: {row[1]}")

cur.close()
conn.close()

PULocationID: 74, num_trips: 81
PULocationID: 74, num_trips: 72
PULocationID: 74, num_trips: 71


## Question 6. Tumbling window - largest tip

Create a Flink job that uses a 1-hour tumbling window to compute the
total `tip_amount` per hour (across all locations).

Which hour had the highest total tip amount?

- 2025-10-01 18:00:00
- 2025-10-16 18:00:00
- 2025-10-22 08:00:00
- 2025-10-30 16:00:00

In [27]:
"""CREATE TABLE processed_events_tips (
     window_start TIMESTAMP,
     total_tip DOUBLE PRECISION,
     PRIMARY KEY (window_start)
 );"""

'CREATE TABLE processed_events_tips (\n     window_start TIMESTAMP,\n     total_tip DOUBLE PRECISION,\n     PRIMARY KEY (window_start)\n );'

In [ ]:
#!docker exec 07-streaming-jobmanager-1 flink run -py /opt/src/job/tips_job.py

In [31]:
conn = psycopg2.connect(
    host="localhost",
    port=5432,
    dbname="postgres",
    user="postgres",
    password="postgres"
)

cur = conn.cursor()
cur.execute("""
    SELECT window_start, total_tip
    FROM processed_events_tips
    ORDER BY total_tip DESC
    LIMIT 3;
""")

rows = cur.fetchall()
for row in rows:
    print(f"PULocationID: {row[0]}, total_tip: {row[1]}")

cur.close()
conn.close()

PULocationID: 2025-10-16 18:00:00, total_tip: 524.9599999999998
PULocationID: 2025-10-30 16:00:00, total_tip: 507.1
PULocationID: 2025-10-10 17:00:00, total_tip: 499.6000000000002
